# 10. Cards for every season (2021-22 to 2025-26)

Notebook 08 built the full card table for 2025-26 only. `player_card.build_season_cards` packages the same steps (WAR v0.4, card metrics, percentiles, NHL EDGE, age and draft) so they can run on any season.

This notebook:
1. downloads NHL EDGE data for 2021-22 to 2024-25 (saved files are skipped, so it can be stopped and rerun)
2. downloads NHL profiles for any player not saved yet
3. builds each season's full card table
4. checks that the rebuilt 2025-26 table matches the saved one on every card column
5. writes one combined slim table for the app, `app_data/cards.csv`
6. prints a few sanity checks

`season` is MoneyPuck's start year: 2025 = 2025-26. The app displays `season_label`, never the integer.

## Setup

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append("../src")
from war_model import run_season, V04_PARAMS
from edge_data import download_season, download_bios
from player_card import build_season_cards, export_app_table, app_columns, season_label

DATA = Path("../data")
APP_DATA = Path("../app_data")
BIO_DIR = DATA / "bio_raw"                 # player profiles, not season-specific
REFERENCE = DATA / "cards_2025-26.csv"     # fixed reference for the regression check; never overwritten

SEASONS = [2021, 2022, 2023, 2024, 2025]   # MoneyPuck start years: 2025 = 2025-26
NEW_SEASONS = SEASONS[:-1]                 # 2025-26 is already downloaded (notebook 08)


def skaters_file(season):
    return DATA / f"skaters{season_label(season)[2:]}.csv"     # 2021 -> skaters21-22.csv


def edge_dir(season):
    return DATA / "edge_raw" / season_label(season)            # 2021 -> edge_raw/2021-22

## 1. Download NHL EDGE data

About 3,800 player-seasons, roughly 40 minutes the first time. Each player is saved as its own file, so if this stops, run the cell again and it picks up where it left off.

In [2]:
player_ids = {s: run_season(skaters_file(s), V04_PARAMS)["playerId"].tolist() for s in SEASONS}

for s in NEW_SEASONS:
    print(f"\n{season_label(s)}: {len(player_ids[s])} skaters")
    download_season(player_ids[s], f"{s}{s + 1}", edge_dir(s))


2021-22: 1003 skaters
0 already saved, 1003 to download (about 10 minutes)
  50 / 1003
  100 / 1003
  150 / 1003
  200 / 1003
  250 / 1003
  300 / 1003
  350 / 1003
  400 / 1003
  450 / 1003
  500 / 1003
  550 / 1003
  600 / 1003
  650 / 1003
  700 / 1003
  750 / 1003
  800 / 1003
  850 / 1003
  900 / 1003
  950 / 1003
  1000 / 1003
  1003 / 1003
Done, no failures.

2022-23: 951 skaters
0 already saved, 951 to download (about 10 minutes)
  50 / 951
  100 / 951
  150 / 951
  200 / 951
  250 / 951
  300 / 951
  350 / 951
  400 / 951
  450 / 951
  500 / 951
  550 / 951
  600 / 951
  650 / 951
  700 / 951
  750 / 951
  800 / 951
  850 / 951
  900 / 951
  950 / 951
  951 / 951
Done, no failures.

2023-24: 924 skaters
0 already saved, 924 to download (about 9 minutes)
  50 / 924
  100 / 924
  150 / 924
  200 / 924
  250 / 924
  300 / 924
  350 / 924
  400 / 924
  450 / 924
  500 / 924
  550 / 924
  600 / 924
  650 / 924
  700 / 924
  750 / 924
  800 / 924
  850 / 924
  900 / 924
  924 / 924

## 2. Download player profiles (age and draft)

Profiles aren't season-specific, so only players not saved yet are downloaded.

In [3]:
all_ids = sorted(set().union(*player_ids.values()))
print(len(all_ids), "different players across the five seasons")
failed_bios = download_bios(all_ids, BIO_DIR)

1480 different players across the five seasons
940 already saved, 540 to download (about 5 minutes)
  50 / 540
  100 / 540
  150 / 540
  200 / 540
  250 / 540
  300 / 540
  350 / 540
  400 / 540
  450 / 540
  500 / 540
  8482824: The read operation timed out
  8482829: The read operation timed out
  540 / 540
Done, no failures.


## 3. Build each season's card table

The four older seasons are saved to `data/cards_<label>.csv` (gitignored). 2025-26 is checked in step 4 before anything is saved.

In [4]:
for s in SEASONS:
    n_files = len(list(edge_dir(s).glob("*.json")))
    assert n_files >= len(player_ids[s]), f"{season_label(s)}: only {n_files} of {len(player_ids[s])} EDGE files. Rerun step 1."

cards = {s: build_season_cards(skaters_file(s), edge_dir(s), BIO_DIR) for s in SEASONS}

for s in NEW_SEASONS:
    path = DATA / f"cards_{season_label(s)}.csv"
    cards[s].to_csv(path, index=False)
    print(f"{season_label(s)}: {len(cards[s])} players, {int(cards[s]['has_edge'].sum())} with EDGE data -> data/{path.name}")

2021-22: 1003 players, 1001 with EDGE data -> data/cards_2021-22.csv
2022-23: 951 players, 950 with EDGE data -> data/cards_2022-23.csv
2023-24: 924 players, 924 with EDGE data -> data/cards_2023-24.csv
2024-25: 920 players, 918 with EDGE data -> data/cards_2024-25.csv


## 4. Regression check: rebuilt 2025-26 vs. the saved reference

`data/cards_2025-26.csv` (from notebook 08) is the fixed reference and is never overwritten. The rebuilt table must match it on `season` and every column the card draws: numbers within 1e-9, text and True/False exactly. If it doesn't, the `assert` stops here and nothing is saved.

In [5]:
def differences(ref, new, cols):
    """Columns where ref and new disagree, with up to 3 example playerIds each."""
    a = ref.set_index("playerId").sort_index()
    b = new.set_index("playerId").sort_index()
    out = {}
    for c in cols:
        x, y = a[c], b[c]
        numeric = pd.api.types.is_numeric_dtype(x) and pd.api.types.is_numeric_dtype(y) and not pd.api.types.is_bool_dtype(x)
        if numeric:
            same = np.isclose(x.astype(float), y.astype(float), rtol=0, atol=1e-9, equal_nan=True)
        else:
            same = np.asarray((x == y) | (x.isna() & y.isna()))
        if not same.all():
            out[c] = a.index[~same][:3].tolist()
    return out


reference = pd.read_csv(REFERENCE)
rebuilt = cards[2025]
assert set(reference["playerId"]) == set(rebuilt["playerId"]), "Rebuilt 2025-26 has different players than the reference"

card_cols = ["season"] + app_columns()[1:]          # every card column except the playerId key
bad = differences(reference, rebuilt, card_cols)
assert not bad, f"Rebuilt 2025-26 does not match the reference on: {bad}"
print(f"Regression check passed: {len(card_cols)} card columns match for {len(rebuilt)} players.")

other = [c for c in reference.columns if c in rebuilt.columns and c not in card_cols and c != "playerId"]
print("Other shared columns that differ (info only):", differences(reference, rebuilt, other) or "none")

rebuilt.to_csv(DATA / "cards_2025-26_v2.csv", index=False)
print("Saved data/cards_2025-26_v2.csv. The reference file was not changed.")

Regression check passed: 57 card columns match for 940 players.
Other shared columns that differ (info only): none
Saved data/cards_2025-26_v2.csv. The reference file was not changed.


## 5. Combined slim table for the app

One row per player per season, only the columns the card draws plus `season` and `season_label`. Built from the tables in memory above.

In [6]:
combined = pd.concat([cards[s] for s in SEASONS], ignore_index=True)
slim = export_app_table(combined, APP_DATA / "cards.csv", by_season=True)

# No raw MoneyPuck situation columns, except ev_rel_xga_60 (shown on the card)
raw = [c for c in slim.columns if c.startswith(("ev_", "pp_", "pk_", "all_")) and c != "ev_rel_xga_60"]
assert not raw, f"Raw situation columns in slim table: {raw}"

# Hidden card rows are left out
assert "oz_pct_ev" not in slim.columns and "pct_oz_pct_ev" not in slim.columns

# One row per player per season, the expected columns, all five seasons
assert not slim.duplicated(["playerId", "season"]).any(), "A player appears twice in one season"
assert list(slim.columns) == app_columns(by_season=True)
assert sorted(slim["season"].unique()) == SEASONS

print("All checks passed.", slim.shape, "-> app_data/cards.csv")

All checks passed. (4738, 59) -> app_data/cards.csv


## 6. Sanity checks

In [7]:
print(slim.groupby("season_label").size().rename("player-seasons").to_string())

show = ["season_label", "team", "games_played", "war"]
for name in ["Connor McDavid", "Macklin Celebrini"]:
    print(f"\n{name}")
    print(slim[slim["name"] == name][show].round(2).to_string(index=False))

assert sorted(slim.loc[slim["name"] == "Connor McDavid", "season"]) == SEASONS, "McDavid should have all five seasons"
assert sorted(slim.loc[slim["name"] == "Macklin Celebrini", "season"]) == [2024, 2025], "Celebrini should have 2024-25 and 2025-26 only"

season_label
2021-22    1003
2022-23     951
2023-24     924
2024-25     920
2025-26     940

Connor McDavid
season_label team  games_played  war
     2021-22  EDM            80 6.06
     2022-23  EDM            82 8.74
     2023-24  EDM            76 5.99
     2024-25  EDM            67 4.56
     2025-26  EDM            82 6.12

Macklin Celebrini
season_label team  games_played  war
     2024-25  SJS            70 2.51
     2025-26  SJS            82 4.71
